# PhishGuard AI — DistilBERT Fine-Tuning (Google Colab GPU)

**Steps:**
1. Run Cell 1 — check GPU
2. Run Cell 2 — install deps
3. Run Cell 3 — upload `data/processed/emails.csv`
4. Run Cell 4 — train (~5–10 min on T4)
5. Run Cell 5 — download `distilbert-phishing.zip`
6. Unzip into `models/distilbert-phishing/` in your local project

> **Runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Cell 1: Verify GPU ───────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
# ── Cell 3: Upload emails.csv ─────────────────────────────────────────────────
# When the file picker appears, navigate to:
#   PhishGuard AI/data/processed/emails.csv

import os
from google.colab import files

os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Select emails.csv from your local data/processed/ folder:')
uploaded = files.upload()

for fname in uploaded:
    os.rename(fname, 'data/processed/emails.csv')

import pandas as pd
df = pd.read_csv('data/processed/emails.csv')
print(f'\nLoaded {len(df):,} rows')
print('Label distribution:', df['label'].value_counts().to_dict())

In [ ]:
# ── Cell 4: Fine-tune DistilBERT ──────────────────────────────────────────────
# Trains on the FULL dataset (158k rows) — ~10 min on T4 GPU

import re
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ── Config ───────────────────────────────────────────────────────────────────
BASE_MODEL   = 'distilbert-base-uncased'
OUTPUT_DIR   = 'models/distilbert-phishing'
MAX_LENGTH   = 128
EPOCHS       = 3
BATCH_TRAIN  = 32   # T4 has 16GB VRAM — 32 fits comfortably
BATCH_EVAL   = 64
LEARNING_RATE = 2e-5
LABEL2ID = {'safe': 0, 'phishing': 1}
ID2LABEL = {0: 'safe', 1: 'phishing'}

# ── Text cleaning (mirrors src/ml_model.py) ───────────────────────────────────
def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+', ' URL ', text)
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

# ── Load & prepare data ───────────────────────────────────────────────────────
print('Loading dataset...')
df = pd.read_csv('data/processed/emails.csv')[['text', 'label']].dropna()
df['text']  = df['text'].astype(str)
df['label'] = df['label'].astype(int)
print(f'  {len(df):,} rows | phishing={df["label"].sum():,} | safe={(df["label"]==0).sum():,}')

dataset = Dataset.from_pandas(df, preserve_index=False)
dataset = dataset.train_test_split(test_size=0.20, seed=42)
print(f'  Train: {len(dataset["train"]):,} | Test: {len(dataset["test"]):,}')

# ── Tokenise ──────────────────────────────────────────────────────────────────
print('\nLoading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

print('Tokenising...')
dataset = dataset.map(tokenize, batched=True, batch_size=512, remove_columns=['text'])
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# ── Model ─────────────────────────────────────────────────────────────────────
print('\nLoading base model...')
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# ── Training args ─────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    learning_rate=LEARNING_RATE,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=100,
    report_to='none',
    fp16=True,   # half-precision — 2x faster on GPU, no accuracy loss
)

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    preds = (probs[:, 1] >= 0.5).astype(int)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, zero_division=0)),
        'roc_auc':  float(roc_auc_score(labels, probs[:, 1])),
    }

# ── Train ─────────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    compute_metrics=compute_metrics,
)

print('\nStarting fine-tuning...')
trainer.train()

# ── Save ──────────────────────────────────────────────────────────────────────
print('\nSaving model...')
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ── Final eval ────────────────────────────────────────────────────────────────
metrics = trainer.evaluate()
print('\n══════════════ FINAL RESULTS ══════════════')
for k, v in metrics.items():
    print(f'  {k:<30s} {v:.4f}')
print('============================================')
print(f'\nModel saved to: {OUTPUT_DIR}')

In [ ]:
# ── Cell 5: Download trained model ───────────────────────────────────────────
# This zips models/distilbert-phishing/ and downloads it to your machine.
# Then unzip it into PhishGuard AI/models/distilbert-phishing/

import shutil
from google.colab import files

print('Zipping model...')
shutil.make_archive('distilbert-phishing', 'zip', 'models', 'distilbert-phishing')

size_mb = os.path.getsize('distilbert-phishing.zip') / 1e6
print(f'Downloading distilbert-phishing.zip ({size_mb:.0f} MB)...')
files.download('distilbert-phishing.zip')
print('Done! Unzip into PhishGuard AI/models/distilbert-phishing/')

In [ ]:
# ── Cell 6 (optional): Quick inference test ───────────────────────────────────
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0,
)

test_emails = [
    'URGENT: Your PayPal account has been suspended. Click here to verify immediately.',
    'Hi team, just a reminder about the standup at 10am tomorrow.',
    'Congratulations! You have won $2,500,000. Claim your prize now via Western Union.',
    'Your GitHub pull request #247 has been merged into main.',
]

print(f'{"Email":<60} {"Label":<12} {"Score"}')
print('-' * 82)
for email in test_emails:
    result = classifier(email, truncation=True, max_length=128)[0]
    print(f'{email[:59]:<60} {result["label"]:<12} {result["score"]*100:.1f}%')